## Data Preprocessing

This stage prepares the raw London crime dataset for machine learning. The preprocessing pipeline removes duplicate rows, drops non-predictive identifiers, standardises column names, imputes missing geographic coordinates, encodes the target labels, converts categorical features into numerical format, scales numerical variables, and reshapes the feature matrix for use with a Conv1D neural network.

### Remove Duplicate Records and Drop the Crime Identifier

Duplicate records are removed to prevent repeated samples from biasing the model. The `Crime ID` column is removed because it represents a unique identifier rather than a meaningful predictive feature.

In [ ]:
print(df.columns)

df.drop_duplicates(inplace=True)
df.drop(columns=["Crime ID"], inplace=True)

### Standardise Column Names

Column names are stripped of leading and trailing whitespace to ensure consistent referencing of variables throughout the preprocessing pipeline.

In [ ]:
df.columns = df.columns.str.strip()

### Impute Missing Geographic Coordinates

The dataset contains missing values in the geographic coordinate fields `Longitude` and `Latitude`. Missing values are filled using the median coordinates within each `LSOA name` group. If a geographic group contains no coordinate values, a global median fallback is applied.

In [ ]:
numeric_cols = ['Longitude', 'Latitude']
group_col = 'LSOA name'

group_medians = df.groupby(group_col)[numeric_cols].median()

df = df.merge(
    group_medians,
    left_on=group_col,
    right_index=True,
    how='left',
    suffixes=('', '_grp')
)

for col in numeric_cols:
    df[col] = df[col].fillna(df[f'{col}_grp'])

global_medians = df[numeric_cols].median()
df[numeric_cols] = df[numeric_cols].fillna(global_medians)

df.drop(columns=[f'{c}_grp' for c in numeric_cols], inplace=True)

### Encode Target Variable

The target variable `Crime type` is encoded into numerical labels using `LabelEncoder`. This transformation allows categorical crime types to be used in supervised learning models.

In [ ]:
target_col = 'Crime type'
encoder = LabelEncoder()
df['target_encoded'] = encoder.fit_transform(df[target_col])
num_classes = df['target_encoded'].nunique()

### Construct Feature Matrix

Two numerical features (`Longitude`, `Latitude`) are retained directly. Selected categorical features (`Reported by`, `Falls within`) are converted using one-hot encoding so that the model can process them as numerical inputs.

In [ ]:
numeric_cols = ['Longitude', 'Latitude']
categorical_cols = ['Reported by', 'Falls within']

df_encoded = pd.get_dummies(df[categorical_cols])

X = pd.concat([df[numeric_cols], df_encoded], axis=1).values
y = to_categorical(df['target_encoded'], num_classes=num_classes)

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")

### Standardise Numerical Features

The numerical coordinate features are standardised using `StandardScaler` so that they have zero mean and unit variance. Standardisation improves optimisation stability during neural network training.

In [ ]:
scaler = StandardScaler()
X[:, :len(numeric_cols)] = scaler.fit_transform(X[:, :len(numeric_cols)])

X = X.astype("float32")

### Reshape Input for Conv1D

The feature matrix is reshaped into a three-dimensional tensor of shape `(samples, timesteps, features)` to match the input format required by Conv1D neural networks.

In [ ]:
X = X.reshape(X.shape[0], X.shape[1], 1)
print(f"Reshaped X for Conv1D: {X.shape}")